In [ ]:
import os
import sys
sys.path.insert(1,r'D:\Khabarov\Репозиторий\sql_premises_and_volumes')
sys.path.insert(1,r'D:\Khabarov\Репозиторий\sql_premises_and_volumes\Helpers')
from operator import index
from tokenize import group

import numpy as np
import pandas as pd


from Helpers.DbConnector import DbConnector
from Helpers.PremiseHelper import PremiseHelper
from Helpers.ParamsAndFuns import ParamsAndFuns as p
from Access.AccessInfo import AccessInfo as ai
from Helpers.MultiPremiseHelper import MultiPremiseHelper

desired_width=320
pd.set_option('display.width', desired_width)
np.set_printoptions(linewidth=desired_width)
pd.set_option('display.max_columns',10)
res = ''

# coId= 'e5089524-98ff-4b2d-89d7-8a3800c1ce33'
# stage = 'Стадия П'
modelType = 'premise'
shortName = ai.short_name_prem
version = 999
full_path = ai.full_path_prem

source_path = r'D:\Khabarov\Репозиторий\sql_premises_and_volumes\SourceData\ИсходныеДанные.xlsx'

dbCon = DbConnector()
multi_prems = MultiPremiseHelper(source_path,dbCon)

In [ ]:
#Получаем окна
def get_df_windows_with_premises(data):
    df = data[data[p.bru_destination_pn].isin(['Витраж','Окно']) == True]
    df = df[(df[p.premise_part_number_to_pn].str.contains("Ж") == True)
            | (df[p.premise_part_number_from_pn].str.contains("Ж") == True)]
    windows = df[['Наименование ОС',p.premise_part_number_to_pn,p.premise_part_number_from_pn]]

    #Получаем комнаты
    lp_rooms = ['Лоджия'
                ,'Лоджия (холодная)'
                ,'Лоджия (техническая)'
                ,'Балкон'
                ,'Терраса'
                ,'Терраса на земле']
    df = data[(data[p.bru_destination_pn] == "Жилье")
                    & (data[p.name_pn].isin(lp_rooms) == False)]
    rooms = df[["Наименование ОС",p.premise_part_number,p.name_pn]]

    #Присоединяем инфу по помещениям To
    rooms_to = rooms
    rooms_to = rooms_to.rename(mapper={p.premise_part_number:p.premise_part_number_to_pn
                            ,p.name_pn: "Имя To"},axis=1)
    res_df = pd.merge(left=windows,right=rooms_to,on=["Наименование ОС",p.premise_part_number_to_pn],how='left')

    #Получаем инфу по помещениям From
    rooms_from = rooms
    rooms_from = rooms_from.rename(mapper={p.premise_part_number:p.premise_part_number_from_pn
                            ,p.name_pn: "Имя From"},axis=1)
    wnd_with_rooms = pd.merge(left=res_df,right=rooms_from,on=["Наименование ОС",p.premise_part_number_from_pn],how='left')
    wnd_with_rooms['Имя и номер To'] = wnd_with_rooms['Номер части помещения To'].astype(str) + " - " + wnd_with_rooms['Имя To'].astype(str)
    wnd_with_rooms['Имя и номер From'] = wnd_with_rooms['Номер части помещения From'].astype(str) + " - " + wnd_with_rooms['Имя From'].astype(str)


    #Получаем количество уникальных помещений
    res_df = wnd_with_rooms.groupby("Наименование ОС",as_index=False).agg(N_окон=("Наименование ОС",'size')
                                                ,Num_помещений_To=('Имя и номер To',lambda x: x)
                                                ,Num_помещений_From=('Имя и номер From',lambda x: x)
                                                )
    res_df['Num_помещений'] = res_df.apply(lambda x: np.hstack((x['Num_помещений_To'],x['Num_помещений_From'])),axis=1)
    res_df["Num_помещений_Уник"] = res_df["Num_помещений"].apply(lambda x: set(list(filter(lambda t: not('nan' in t),x))))
    res_df['N_помещений'] = res_df["Num_помещений_Уник"].apply(lambda x: len(x))

    return res_df


In [ ]:
import os
import sys
sys.path.insert(1, r'D:\Khabarov\Репозиторий\sql_premises_and_volumes')
from Helpers.ParamsAndFuns import ParamsAndFuns as p
import numpy as np
import pandas as pd
from Helpers.VolumesHelper import VolumesHelper
p.set_np_pd_opts()


source_path = r'D:\Khabarov\Репозиторий\sql_premises_and_volumes\SourceData\ИсходныеДанные.xlsx' #Здесь укажи путь к табл
volHel = VolumesHelper(source_path=source_path)
dfFull = volHel.fullDf

#=====Запусти для загрузки данных=====

In [ ]:
dfFull

params = ['Наименование ОС','Стадия','Имя СК','Тип','Вид','Размер','Производитель','Часть системы','Секция','Этаж']
rads = dfFull[dfFull['Тип'] == 'Радиатор']
rads_count = rads.groupby(params,as_index=False).agg(N_рад=('Наименование ОС','size'))
rads_count = rads_count.sort_values(['Наименование ОС','Секция','Этаж'])

# final_df = pd.merge(left=res_df,right=rads_count,on='Наименование ОС',how='left')
# final_df = final_df.rename(mapper={'Наименование ОС':"Объект строительства"
#                                    ,"N_окон": "Кол-во окон и витражей, шт"
#                                    ,"N_помещений": "Кол-во помещений с окном (кроме лп), шт"
#                                    ,"N_рад": "Кол-во радиаторов, шт"
#                                    }
#                                    ,axis=1)
# final_df = final_df[["Объект строительства"
#                      ,"Кол-во окон и витражей, шт"
#                      ,"Кол-во помещений с окном (кроме лп), шт"
#                      ,"Кол-во радиаторов, шт"
#                      ]]
# # final_df.to_excel('D:\Khabarov\Репозиторий\sql_premises_and_volumes\Data\ОкнаРадиаторы.xlsx',sheet_name='Лист1',index=False)
# final_df
rads_count.to_excel('D:\Khabarov\Репозиторий\sql_premises_and_volumes\Data\Радиаторы.xlsx',sheet_name='Лист1',index=False)